# ANON AI ULTRA — Ministral 3 temel model testi

Bu notebook, ANON AI için seçtiğimiz **Ministral 3 3B Instruct** temel modelini Colab GPU üzerinde yükler ve Türkçe sohbet testlerini çalıştırır.

- Model: `mistralai/Ministral-3-3B-Instruct-2512-BF16`
- Lisans: Apache 2.0 (model kartı)
- Bu aşama yalnızca temel modelin yüklenmesini ve Türkçe cevap kalitesini doğrular.
- Henüz LoRA eğitimi yapmaz; önce modelin Colab GPU'da sorunsuz çalıştığını görelim.
- Android uygulamasına bağlamaz; mevcut çalışan uygulama modelden bağımsız kalır.

Resmî model kartı: https://huggingface.co/mistralai/Ministral-3-3B-Instruct-2512-BF16

In [ ]:
import sys, subprocess, torch, os
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "GPU bulunamadı")
assert torch.cuda.is_available(), "Colab > Runtime > Change runtime type > GPU seç."
print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "accelerate", "safetensors", "mistral-common"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "git+https://github.com/huggingface/transformers"], check=True)
print("Kurulum tamam. Transformers sürümü:")
import transformers
print(transformers.__version__)

## 1. Modeli ve tokenizer'ı yükle

Bu model ailesi için Mistral'ın kendi tokenizer backend'i ve Transformers'ın güncel ana dal desteği kullanılır. Model ağırlıkları FP16 olarak yüklenir; GPU belleği yetmezse Colab oturumunu daha yüksek belleğe sahip GPU ile yeniden başlatmak gerekebilir.

In [ ]:
import torch
from transformers import Mistral3ForConditionalGeneration, MistralCommonBackend

MODEL_ID = "mistralai/Ministral-3-3B-Instruct-2512-BF16"
tokenizer = MistralCommonBackend.from_pretrained(MODEL_ID)
model = Mistral3ForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.eval()
print("Model başarıyla yüklendi:", MODEL_ID)
print("Tokenizer hazır.")

## 2. Sohbet fonksiyonu

Yalnızca metin mesajı gönderiyoruz. Görsel girişi bu ilk testin kapsamı dışında.

In [ ]:
def anon_ai_test_chat(question, max_new_tokens=128):
    messages = [{"role": "user", "content": question}]
    inputs = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        return_dict=True,
    )
    inputs = {k: (v.to(model.device) if torch.is_tensor(v) else v) for k, v in inputs.items()}
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
    prompt_len = inputs["input_ids"].shape[-1]
    answer = tokenizer.decode(output[0][prompt_len:], skip_special_tokens=True)
    return answer.strip()

print(anon_ai_test_chat("Merhaba, nasılsın?"))

## 3. Türkçe kalite testi

Cevapları incele. Bu sonuçlar temel modelin performansını gösterir; ANON AI'ye özel ince ayar henüz yapılmadı.

In [ ]:
TEST_QUESTIONS = [
    "Merhaba, nasılsın? Kısaca cevap ver.",
    "Türkiye'nin başkenti neresidir? Sadece şehir adını ve kısa açıklamasını yaz.",
    "Yapay zekâ nedir? Bunu 12 yaşındaki birine anlatır gibi açıkla.",
    "Almanca 'Merhaba, nasılsın?' nasıl söylenir? Türkçe anlamını da yaz.",
    "Bana kendini tanıt. Adın ANON AI olsun ve Türkçe konuş.",
    "Bir sayının karesini nasıl hesaplarım? 7 örneğiyle göster.",
]
results = []
for i, question in enumerate(TEST_QUESTIONS, 1):
    print(f"\n{'='*60}\nTEST {i}: {question}")
    answer = anon_ai_test_chat(question, max_new_tokens=160)
    results.append({"question": question, "answer": answer})
    print("ANON AI:", answer)

## 4. Test raporunu kaydet

Çıktıyı indirip sonraki adımda paylaşabilirsin. Model düzgün cevap verirse Türkçe veri setini temizleyip LoRA eğitim notebook'unu hazırlayacağız.

In [ ]:
import json
REPORT_PATH = "/content/ANON-AI-ULTRA-Ministral3-test-report.json"
with open(REPORT_PATH, "w", encoding="utf-8") as f:
    json.dump({
        "model_id": MODEL_ID,
        "gpu": torch.cuda.get_device_name(0),
        "results": results,
    }, f, ensure_ascii=False, indent=2)
print("Rapor hazır:", REPORT_PATH)
from google.colab import files
files.download(REPORT_PATH)